In [ ]:
import os
from pathlib import Path
import random
import shutil
import numpy as np
import tensorflow as tf
import seaborn as sns
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models
from sklearn.metrics import confusion_matrix, classification_report

os.environ["KAGGLE_USERNAME"] = "your_username"
os.environ["KAGGLE_KEY"] = "your_api_key"

!pip install -q kaggle
!kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset -p /content --unzip

In [ ]:
random.seed(42)

src_train = Path("/content/Training")
src_test  = Path("/content/Testing")
dst_root  = Path("/content/data")
dst_train, dst_val, dst_test = dst_root/"train", dst_root/"val", dst_root/"test"

if dst_root.exists():
    shutil.rmtree(dst_root)
shutil.copytree(src_test, dst_test)

VAL_FRAC = 0.15
classes  = sorted([d.name for d in src_train.iterdir() if d.is_dir()])

for c in classes:
    imgs = [p for p in (src_train / c).glob("*") if p.is_file()]
    random.shuffle(imgs)
    n_val = int(len(imgs) * VAL_FRAC)
    (dst_train / c).mkdir(parents=True, exist_ok=True)
    (dst_val   / c).mkdir(parents=True, exist_ok=True)
    for p in imgs[:n_val]:
        shutil.copy2(p, dst_val   / c / p.name)
    for p in imgs[n_val:]:
        shutil.copy2(p, dst_train / c / p.name)

for split in ["train", "val", "test"]:
    base = dst_root / split
    print(split, {c.name: len(list(c.glob("*"))) for c in base.iterdir()})

In [ ]:
IMG_SIZE = (256, 256)
BATCH    = 32
SEED     = 42
AUTOTUNE = tf.data.AUTOTUNE

print("TF version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

kw = dict(label_mode="categorical", image_size=IMG_SIZE, batch_size=BATCH)
train_ds    = tf.keras.utils.image_dataset_from_directory(str(dst_train), shuffle=True, seed=SEED, **kw)
val_ds      = tf.keras.utils.image_dataset_from_directory(str(dst_val),   shuffle=False, **kw)
test_ds     = tf.keras.utils.image_dataset_from_directory(str(dst_test),  shuffle=False, **kw)
class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:", class_names)

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)
test_ds  = test_ds.prefetch(AUTOTUNE)

In [ ]:
def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, 0.1)
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

def normalize(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

train_ds_cnn = train_ds.map(augment)
val_ds_cnn   = val_ds.map(normalize)
test_ds_cnn  = test_ds.map(normalize)

In [ ]:
def build_custom_cnn(num_classes, input_shape=(256, 256, 3)):
    inputs = layers.Input(shape=input_shape)

    for filters in [64, 128, 256, 512]:
        x = layers.Conv2D(filters, (3, 3), padding="same")(inputs if filters == 64 else x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = layers.MaxPooling2D((2, 2))(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return models.Model(inputs, outputs)

custom_cnn = build_custom_cnn(num_classes)
custom_cnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
custom_cnn.summary()

In [ ]:
callbacks_cnn = [
    tf.keras.callbacks.ModelCheckpoint("custom_cnn_best.keras", monitor="val_accuracy", save_best_only=True, mode="max"),
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=3),
]

history_cnn = custom_cnn.fit(
    train_ds_cnn,
    validation_data=val_ds_cnn,
    epochs=50,
    callbacks=callbacks_cnn,
)

In [ ]:
test_loss_cnn, test_acc_cnn = custom_cnn.evaluate(test_ds_cnn, verbose=1)
print(f"loss={test_loss_cnn:.4f}  acc={test_acc_cnn:.4f}")

y_true = np.concatenate([np.argmax(y.numpy(), axis=1) for _, y in test_ds_cnn])
y_pred = np.argmax(custom_cnn.predict(test_ds_cnn, verbose=1), axis=1)

cm = confusion_matrix(y_true, y_pred)
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=class_names, yticklabels=class_names, cmap="Blues")
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix – Custom CNN")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (tr, val, title) in zip(axes, [
    ("loss",     "val_loss",     "Loss"),
    ("accuracy", "val_accuracy", "Accuracy"),
]):
    ax.plot(history_cnn.history[tr],  label="Train")
    ax.plot(history_cnn.history[val], label="Val")
    ax.legend(); ax.set_title(title)
plt.tight_layout(); plt.show()

In [ ]:
custom_cnn.save("custom_cnn_best.keras")
print(os.listdir())

In [ ]:
from google.colab import files
files.download("custom_cnn_best.keras")